# Notebook-legal.com.br

[Dataset](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

In [2]:
%pip install pandas duckdb pyarrow


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import duckdb
from time import perf_counter
import datetime

In [4]:
arquivo = "yellow_tripdata_2026-01.parquet"

tempo_pandas = perf_counter()
df = pd.read_parquet(arquivo)
print(f"Pandas: {perf_counter() - tempo_pandas:.5f}s")

con = duckdb.connect()

tempo_duckdb = perf_counter()
con.execute(f"""
    CREATE TABLE corridas AS
    SELECT * FROM read_parquet ('{ arquivo }')
""")
print(f"DuckDB: {perf_counter() - tempo_duckdb:.5f}s")

razao = tempo_pandas / tempo_duckdb

print(f"\n\n{'DuckDB' if razao > 1 else 'Pandas'} foi {max(razao, 1/razao):.5f}x mais rápido")

Pandas: 1.23737s
DuckDB: 0.66055s


Pandas foi 1.00036x mais rápido


In [5]:
print(f"Número de Linhas no df: {len(df)}")

con.sql("SELECT COUNT(*) FROM corridas").show()

print(f"Colunas: {df.columns}")

con.sql("""
    SELECT column_name
    FROM information_schema.columns
    WHERE table_name = 'corridas'
""").show()

Número de Linhas no df: 3724889
┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      3724889 │
└──────────────┘

Colunas: Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'Airport_fee',
       'cbd_congestion_fee'],
      dtype='str')
┌───────────────────────┐
│      column_name      │
│        varchar        │
├───────────────────────┤
│ VendorID              │
│ tpep_pickup_datetime  │
│ tpep_dropoff_datetime │
│ passenger_count       │
│ trip_distance         │
│ RatecodeID            │
│ store_and_fwd_flag    │
│ PULocationID          │
│ DOLocationID          │
│ payment_type          │
│ fare_amount           │
│ extra                 │
│ mta_tax               │
│ ti

In [6]:
df = df[(df['tpep_pickup_datetime'] >= '2026-01-01') 
      & (df['tpep_dropoff_datetime'] < '2026-02-01')]

con.sql("""
    DELETE FROM corridas
    WHERE tpep_pickup_datetime < '2026-01-01'
    OR tpep_dropoff_datetime >= '2026-02-01';
""")

In [7]:
print(f"Número de Linhas no df: {len(df)}")
con.sql("SELECT COUNT(*) FROM corridas").show()

Número de Linhas no df: 3722601
┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      3722601 │
└──────────────┘



# 1) Corridas por horário (sem IA)
Para cada hora de início da corrida (0 a 23, a partir de tpep_pickup_datetime), calcule o número de corridas, a distância média (trip_distance) e o valor médio pago (total_amount). Ordene pelo número de corridas, do maior para o menor.

In [8]:
tempo_duckdb = perf_counter()
results = con.sql("""
        SELECT date_part('hour', tpep_pickup_datetime) AS hour_pickup, COUNT(*) AS n_corridas, MEAN(trip_distance), MEAN(total_amount)
        FROM corridas
        GROUP BY hour_pickup
        ORDER BY n_corridas DESC
        """).fetchall()
tempo_duckdb = perf_counter() - tempo_duckdb

print(f'DUCKDB -> {tempo_duckdb:.5f}s')
print('(hour, #, mean_distance, mean_price)')
for result in results:
        print(result)

df_ = df.copy()
tempo_pandas = perf_counter()
df_['hour'] = df_['tpep_pickup_datetime'].dt.hour
pandas_results = (
        df_.groupby("hour", as_index=False)
           .agg(
                   n_corridas=('hour', 'size'),
                   mean_distance=('trip_distance', 'mean'),
                   mean_price=('total_amount', 'mean')
           )
           .sort_values('n_corridas', ascending=False)
)
tempo_pandas = perf_counter() - tempo_pandas
print(f'\n\nPANDAS -> {tempo_pandas:.5f}s')
print(pandas_results)

razao = tempo_pandas / tempo_duckdb

print(f"\n\n{'DuckDB' if razao > 1 else 'Pandas'} foi {max(razao, 1/razao):.5f}x mais rápido")

DUCKDB -> 0.02514s
(hour, #, mean_distance, mean_price)
(18, 265574, 3.1910994299140483, 29.317995323335257)
(17, 252172, 6.149088320670021, 30.089567834650595)
(19, 234541, 4.336297193241273, 29.06314810629891)
(16, 227892, 4.936731434188112, 30.85137859161191)
(15, 221677, 5.2552405978067185, 29.340656631042684)
(20, 212883, 5.726825063532561, 29.116928829448693)
(21, 212544, 7.311580425700094, 29.23030765394249)
(14, 207506, 6.578418744518236, 29.0716828429043)
(13, 195708, 5.526827825127233, 27.888121384919003)
(22, 194797, 5.440825269383006, 29.510849397062568)
(12, 187434, 5.2953067746513405, 27.49780498735441)
(11, 171726, 7.003862781407592, 27.032680083388065)
(10, 161418, 9.127193311774398, 27.314342452514182)
(9, 156894, 7.422043545323574, 27.884764363200173)
(23, 150354, 4.015366801016265, 29.996836066881595)
(8, 148280, 8.680417655786359, 29.74172889128643)
(7, 113463, 17.346659968447856, 30.959745467685785)
(0, 111866, 4.750563441975216, 29.563479877711135)
(1, 77818, 8.24

# 4) Consulta livre (com IA)
Utilize IA para propor uma pergunta analítica diferente das anteriores e auxiliar na implementação em Pandas e SQL. Informe a pergunta escolhida e registre os principais prompts utilizados.

In [16]:
def consulta4_pandas(df):
    d = df[
        (df["tpep_pickup_datetime"] >= "2026-01-01")
        & (df["tpep_pickup_datetime"] < "2026-02-01")
        & (df["payment_type"] == 1)
        & (df["fare_amount"] > 0)
        & (df["trip_distance"] > 0)
    ].copy()

    d["faixa_distancia"] = pd.cut(
        d["trip_distance"],
        bins=[0, 2, 5, float("inf")],
        labels=["curta", "media", "longa"],
        right=False,
    )
    d["pct_gorjeta"] = d["tip_amount"] / d["fare_amount"] * 100

    res = (
        d.groupby("faixa_distancia", observed=True)
        .agg(
            n_corridas=("tip_amount", "size"),
            gorjeta_media=("tip_amount", "mean"),
            pct_gorjeta_medio=("pct_gorjeta", "mean"),
        )
        .round(2)
        .reset_index()
        .sort_values("faixa_distancia")
        .reset_index(drop=True)
    )
    res["faixa_distancia"] = res["faixa_distancia"].astype(str)
    return res

tempo_pandas = perf_counter()
res_pd = consulta4_pandas(df)
tempo_pandas = perf_counter() - tempo_pandas
print(res_pd)

SQL_Q4 = """
SELECT
    CASE
        WHEN trip_distance < 2 THEN 'curta'
        WHEN trip_distance < 5 THEN 'media'
        ELSE 'longa'
    END AS faixa_distancia,
    COUNT(*) AS n_corridas,
    ROUND(AVG(tip_amount), 2) AS gorjeta_media,
    ROUND(AVG(tip_amount / fare_amount * 100), 2) AS pct_gorjeta_medio
FROM corridas
WHERE tpep_pickup_datetime >= '2026-01-01'
  AND tpep_pickup_datetime <  '2026-02-01'
  AND payment_type = 1
  AND fare_amount > 0
  AND trip_distance > 0
GROUP BY faixa_distancia
ORDER BY CASE faixa_distancia
             WHEN 'curta' THEN 1
             WHEN 'media' THEN 2
             ELSE 3
         END
"""

tempo_duckdb = perf_counter()
res_sql = con.execute(SQL_Q4).df()
tempo_duckdb = perf_counter() - tempo_duckdb
print(res_sql)

razao = tempo_pandas / tempo_duckdb
print(f"Pandas: {tempo_pandas}s\nDuckDB: {tempo_duckdb}s")
print(f"\n\n{'DuckDB' if razao > 1 else 'Pandas'} foi {max(razao, 1/razao):.5f}x mais rápido")

  faixa_distancia  n_corridas  gorjeta_media  pct_gorjeta_medio
0           curta     1293207           2.79              28.82
1           media      548418           4.15              22.11
2           longa      388915           8.43              16.87
  faixa_distancia  n_corridas  gorjeta_media  pct_gorjeta_medio
0           curta     1293207           2.79              28.82
1           media      548418           4.15              22.11
2           longa      388915           8.43              16.87
Pandas: 0.49299499600010677s
DuckDB: 0.04087801700006821s


DuckDB foi 12.06015x mais rápido


# 5) Corridas simultâneas (sem IA).
Considere as corridas iniciadas no dia 15/01/2026 a partir das 8h. Conte quantos pares de corridas tiveram seus períodos de viagem sobrepostos, considerando os horários de início (tpep_pickup_datetime) e de término (tpep_dropoff_datetime). Considere apenas pares de corridas distintas e conte cada par uma única vez. Execute a consulta para três janelas de tempo, das 8h às 9h, das 8h às 10h e das 8h às 12h (sem incluir o horário final). Informe a quantidade de pares e o tempo de execução de cada janela. Se
alguma execução exceder a memória disponível, registre isso e não aumente a janela.

In [10]:
inicio_execucao = perf_counter()

mascara = (
    (df["tpep_pickup_datetime"] >= "2026-01-15 08:00:00") &
    (df["tpep_pickup_datetime"] < f"2026-01-15 09:00:00")
)
corridas = df.loc[mascara, ["tpep_pickup_datetime", "tpep_dropoff_datetime"]].sort_values("tpep_pickup_datetime").reset_index(drop=True)

inicios = corridas["tpep_pickup_datetime"].to_numpy()
finais = corridas["tpep_dropoff_datetime"].to_numpy()
total = 0
n = len(corridas)

for i in range(n):
    total += ((inicios[i] < finais[i + 1:]) & (inicios[i + 1:] < finais[i])).sum()

print(f"Pandas: 08h–09h | Corridas: {n} | Pares sobrepostos: {total} | Tempo: {perf_counter() - inicio_execucao:.5f}s")


inicio_execucao = perf_counter()

mascara = (
    (df["tpep_pickup_datetime"] >= "2026-01-15 08:00:00") &
    (df["tpep_pickup_datetime"] < f"2026-01-15 10:00:00")
)
corridas = df.loc[mascara, ["tpep_pickup_datetime", "tpep_dropoff_datetime"]].sort_values("tpep_pickup_datetime").reset_index(drop=True)

inicios = corridas["tpep_pickup_datetime"].to_numpy()
finais = corridas["tpep_dropoff_datetime"].to_numpy()
total = 0
n = len(corridas)

for i in range(n):
    total += ((inicios[i] < finais[i + 1:]) & (inicios[i + 1:] < finais[i])).sum()

print(f"Pandas: 08h–10h | Corridas: {n} | Pares sobrepostos: {total} | Tempo: {perf_counter() - inicio_execucao:.5f}s")


inicio_execucao = perf_counter()

mascara = (
    (df["tpep_pickup_datetime"] >= "2026-01-15 08:00:00") &
    (df["tpep_pickup_datetime"] < f"2026-01-15 12:00:00")
)
corridas = df.loc[mascara, ["tpep_pickup_datetime", "tpep_dropoff_datetime"]].sort_values("tpep_pickup_datetime").reset_index(drop=True)

inicios = corridas["tpep_pickup_datetime"].to_numpy()
finais = corridas["tpep_dropoff_datetime"].to_numpy()
total = 0
n = len(corridas)

for i in range(n):
    total += ((inicios[i] < finais[i + 1:]) & (inicios[i + 1:] < finais[i])).sum()

print(f"Pandas: 08h–12h | Corridas: {n} | Pares sobrepostos: {total} | Tempo: {perf_counter() - inicio_execucao:.5f}s")


Pandas: 08h–09h | Corridas: 6603 | Pares sobrepostos: 9907081 | Tempo: 0.14627s
Pandas: 08h–10h | Corridas: 13256 | Pares sobrepostos: 23145124 | Tempo: 0.28625s
Pandas: 08h–12h | Corridas: 25441 | Pares sobrepostos: 46129301 | Tempo: 1.12113s
